# LLMs As Re-Rankers

In [8]:
import pandas as pd


df = pd.read_json('llm-re-ranker-evaluations.jsonl', lines=True, orient='record')

In [19]:
df['re-ranking-family'] = df['llm_for_re_ranking'].apply(lambda i: i.split('-')[0])
df['eval-family'] = df['llm_for_eval'].apply(lambda i: i.split('-')[0])
df['same-family'] = df.apply(lambda i: i['re-ranking-family'] == i['eval-family'], axis=1)

In [26]:
df

,run,dataset,llm_for_re_ranking,llm_for_eval,nDCG@10 Score (original),nDCG@10 Rank (original),nDCG@10 Score (LLM),nDCG@10 Rank (LLM),nDCG@10 Score (LLM - Original),nDCG@10 Rank (Original - LLM),re-ranking-family,eval-family,same-family
0,bm25tuned_rm3_p,trec-dl-2019-judged,AnthropicLLM-claude-3-haiku-20240307-umbrella_...,AnthropicLLM-claude-3-haiku-20240307-umbrella_...,0.616466,22,0.979406,1,0.362940,21,AnthropicLLM,AnthropicLLM,True
1,bm25tuned_rm3_p,trec-dl-2019-judged,AnthropicLLM-claude-3-haiku-20240307-umbrella_...,AnthropicLLM-claude-3-sonnet-20240229-umbrella...,0.616466,22,0.841576,14,0.225110,8,AnthropicLLM,AnthropicLLM,True
2,bm25tuned_rm3_p,trec-dl-2019-judged,AnthropicLLM-claude-3-haiku-20240307-umbrella_...,LiteLLM-llama3-umbrella_zeroshot_basic,0.616466,22,0.825953,12,0.209487,10,AnthropicLLM,LiteLLM,False
3,bm25tuned_rm3_p,trec-dl-2019-judged,AnthropicLLM-claude-3-haiku-20240307-umbrella_...,LiteLLM-llama3.1-umbrella_zeroshot_basic,0.616466,22,0.791299,11,0.174833,11,AnthropicLLM,LiteLLM,False
4,bm25tuned_rm3_p,trec-dl-2019-judged,AnthropicLLM-claude-3-haiku-20240307-umbrella_...,GeminiGPT-gemini-1.5-flash-umbrella_zeroshot_b...,0.616466,22,0.776822,14,0.160356,8,AnthropicLLM,GeminiGPT,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6139,CoRT-standalone,trec-dl-2020-judged,OpenAiGPT-gpt-4o-mini-umbrella_zeroshot_basic,LiteLLM-llama3.1-umbrella_zeroshot_basic,0.696059,28,0.841002,9,0.144944,19,OpenAiGPT,LiteLLM,False
6140,CoRT-standalone,trec-dl-2020-judged,OpenAiGPT-gpt-4o-mini-umbrella_zeroshot_basic,GeminiGPT-gemini-1.5-flash-umbrella_zeroshot_b...,0.696059,28,0.861095,9,0.165036,19,OpenAiGPT,GeminiGPT,False
6141,CoRT-standalone,trec-dl-2020-judged,OpenAiGPT-gpt-4o-mini-umbrella_zeroshot_basic,GeminiGPT-gemini-1.5-flash-8b-umbrella_zerosho...,0.696059,28,0.772730,2,0.076672,26,OpenAiGPT,GeminiGPT,False
6142,CoRT-standalone,trec-dl-2020-judged,OpenAiGPT-gpt-4o-mini-umbrella_zeroshot_basic,OpenAiGPT-gpt-4o-umbrella_zeroshot_basic,0.696059,28,0.845226,9,0.149167,19,OpenAiGPT,OpenAiGPT,True


In [14]:
df['nDCG@10 Score (LLM - Original)'].describe()

count    6144.000000
mean        0.195209
std         0.073703
min         0.037506
25%         0.147727
50%         0.184795
75%         0.231770
max         0.453343
Name: nDCG@10 Score (LLM - Original), dtype: float64

In [20]:
df[df['same-family']]['nDCG@10 Score (LLM - Original)'].describe()

count    1536.000000
mean        0.262832
std         0.091046
min         0.057840
25%         0.180786
50%         0.257974
75%         0.332737
max         0.453343
Name: nDCG@10 Score (LLM - Original), dtype: float64

In [22]:
df[~df['same-family']]['nDCG@10 Score (LLM - Original)'].describe()

count    4608.000000
mean        0.172667
std         0.049482
min         0.037506
25%         0.139641
50%         0.176001
75%         0.205562
max         0.296547
Name: nDCG@10 Score (LLM - Original), dtype: float64

In [24]:
df[df['same-family']]['nDCG@10 Rank (Original - LLM)'].describe()

count    1536.000000
mean       21.072266
std        11.679849
min         0.000000
25%        12.000000
50%        21.000000
75%        31.000000
max        41.000000
Name: nDCG@10 Rank (Original - LLM), dtype: float64

In [15]:
df['nDCG@10 Rank (Original - LLM)'].describe()

count    6144.000000
mean       14.364095
std        10.537180
min        -5.000000
25%         6.000000
50%        11.000000
75%        24.000000
max        41.000000
Name: nDCG@10 Rank (Original - LLM), dtype: float64

In [25]:
df[~df['same-family']]['nDCG@10 Rank (Original - LLM)'].describe()

count    4608.000000
mean       12.128038
std         9.088099
min        -5.000000
25%         5.000000
50%        10.000000
75%        21.000000
max        33.000000
Name: nDCG@10 Rank (Original - LLM), dtype: float64

In [12]:
df['nDCG@10 Score (LLM - Original)'] = df['nDCG@10 Score (LLM)'] - df['nDCG@10 Score (original)']
df['nDCG@10 Rank (Original - LLM)'] = df['nDCG@10 Rank (original)'] - df['nDCG@10 Rank (LLM)']

## Pre-Calculate stuff

In [ ]:
from glob import glob
from trectools import TrecRun, TrecQrel, TrecEval
from tqdm import tqdm
import pandas as pd
import json

DATASET_TO_TREC_IDENTIFIER = {
    'trec-dl-2019-judged': 'trec28',
    'trec-dl-2020-judged': 'trec29',
}

MODEL_TO_NAME = {
    'AnthropicLLM-claude-3-haiku-20240307-umbrella_zeroshot_basic': 'Claude-3-haiku',
    'AnthropicLLM-claude-3-sonnet-20240229-umbrella_zeroshot_basic': 'Claude-3-sonnet',
    'LiteLLM-llama3-umbrella_zeroshot_basic': 'Llama-3',
    'LiteLLM-llama3.1-umbrella_zeroshot_basic': 'Llama-3.1',
    'GeminiGPT-gemini-1.5-flash-umbrella_zeroshot_basic': 'Gemini-1.5-flash',
    'GeminiGPT-gemini-1.5-flash-8b-umbrella_zeroshot_basic': 'Gemini-1.5-flash-8b',
    'OpenAiGPT-gpt-4o-umbrella_zeroshot_basic': 'GPT-4o',
    'OpenAiGPT-gpt-4o-mini-umbrella_zeroshot_basic': 'GPT-4o-mini',
}

RUNS = {}
QRELS = {}

def load_qrels(dataset, name):
    path = f'../data/msmarco-passage-{dataset}/qrels/*.qrels.txt'

    global qrels
    if dataset not in QRELS:
        QRELS[dataset] = {}
        for i in tqdm(glob(path), 'load qrels'):
            qrel_name = i.split('/')[-1].split('.qrels')[0]
            assert qrel_name and qrel_name not in QRELS[dataset]
            QRELS[dataset][qrel_name] = TrecQrel(i)
    
    ret = TrecQrel()
    ret.qrels_data = QRELS[dataset][name].qrels_data.copy()
    return ret

def load_runs(dataset):
    ret = {}

    path = f'../data/trec-system-runs/{DATASET_TO_TREC_IDENTIFIER[dataset]}/deep.passages/input.*.gz'

    global RUNS

    if dataset not in RUNS:
        topics = load_qrels(dataset, 'trec').topics()
        for i in tqdm(glob(path), 'load runs'):
            run_name = i.split('/')[-1].split('.')[1]
            assert run_name and run_name not in ret
            ret[run_name] = TrecRun(i)
            ret[run_name].run_data = ret[run_name].run_data[ret[run_name].run_data['query'].isin(topics)]
            
        RUNS[dataset] = ret
        ret = {}

    for run_name, run in RUNS[dataset].items():
        run_copy = TrecRun()
        run_copy.run_data = run.run_data.copy()
        ret[run_name] = run_copy

    return ret

def re_rank_with_llm(run, dataset, llm_for_re_ranking):
    qrels = load_qrels(dataset, llm_for_re_ranking)
    scores = {}
    for _, i in qrels.qrels_data.iterrows():
        if str(i['query']) not in scores:
            scores[str(i['query'])] = {}
        scores[str(i['query'])][str(i['docid'])] = int(i['rel'])

    ret = TrecRun()
    ret.run_data = run.run_data.copy()
    ret.run_data
    ret.run_data['score'] = ret.run_data.apply(lambda i: scores[str(i['query'])].get(str(i['docid']), -1), axis=1)

    trecformat = ret.run_data.sort_values(["query", "score", "docid"], ascending=[True,False,False]).reset_index()
    topX = trecformat.groupby("query")[["query","docid","score"]].head(1000)
    topX["rank"] = 1
    topX["rank"] = topX.groupby("query")["rank"].cumsum()
    ret.run_data = topX

    return ret

def evaluate(dataset, run_to_modify, llm_for_re_ranking, llm_for_eval):
    qrels = {
        'original': load_qrels(dataset, 'trec'),
        'llm_eval': load_qrels(dataset, llm_for_eval),
    }
    evals = []

    for run_name, run in load_runs(dataset).items():
        evals.append({'run': run_name, 'original': TrecEval(run=run, qrels=qrels['original']).get_ndcg(depth=10), 'llm': TrecEval(run=run, qrels=qrels['llm_eval']).get_ndcg(depth=10)})

        if run_name == run_to_modify:
            modified_run = re_rank_with_llm(run, dataset, llm_for_re_ranking)
            evals.append({'run': run_name + '-re-ranked', 'original': TrecEval(run=modified_run, qrels=qrels['original']).get_ndcg(depth=10), 'llm': TrecEval(run=modified_run, qrels=qrels['llm_eval']).get_ndcg(depth=10)})


    evals = pd.DataFrame(evals)

    evals = evals.sort_values(["original"], ascending=[False]).reset_index()
    evals["rank_original"] = 1
    evals["rank_original"] = evals["rank_original"].cumsum()

    evals = evals.sort_values(["llm"], ascending=[False]).reset_index()
    evals["llm_rank"] = 1
    evals["llm_rank"] = evals["llm_rank"].cumsum()

    re_rank_eval = evals[evals['run'] == run_to_modify + '-re-ranked']
    assert len(re_rank_eval) == 1
    re_rank_eval = re_rank_eval.iloc[0].to_dict()

    return {
        'run': run_to_modify,
        'dataset': dataset,
        'llm_for_re_ranking': llm_for_re_ranking,
        'llm_for_eval': llm_for_eval,
        'nDCG@10 Score (original)': re_rank_eval['original'],
        'nDCG@10 Rank (original)': re_rank_eval['rank_original'],
        'nDCG@10 Score (LLM)': re_rank_eval['llm'],
        'nDCG@10 Rank (LLM)': re_rank_eval['llm_rank'],
    }


In [2]:
for dataset in DATASET_TO_TREC_IDENTIFIER:
    load_runs(dataset)
    load_qrels(dataset, 'trec')


load runs: 100%|██████████| 59/59 [00:35<00:00,  1.67it/s]


In [7]:
def all_test_permutations():
    for dataset in DATASET_TO_TREC_IDENTIFIER:
        for run in load_runs(dataset):
            for evaluation_model in MODEL_TO_NAME:
                for re_rank_model in MODEL_TO_NAME:
                    yield (dataset, run, evaluation_model, re_rank_model)

with open('llm-re-ranker-evaluations.jsonl', 'w') as f:
    for dataset, run, evaluation_model, re_rank_model in tqdm(list(all_test_permutations()), 'Create Re-Rank Evaluations'):
        f.write(json.dumps(evaluate(dataset, run, evaluation_model, re_rank_model)) + '\n')
        f.flush()

Create Re-Rank Evaluations: 100%|██████████| 6144/6144 [4:01:38<00:00,  2.36s/it]  
